# Pretrain experiment

Runs enhanced (unified masked) pretraining and writes everything to
`experiments/<experiment_name>/`:

- `pretrain_config.yaml` — frozen copy of the config used
- `pretrain_metadata.json` — timestamp, git SHA, final/best loss, wallclock
- `pretrain.log` — full training log
- `checkpoints/checkpoint_epoch_*.pth` — saved encoder weights
- `README.md` — name + description (the short note you write below)

Edit the **Parameters** cell, run all cells. Re-running with the same
`experiment_name` will raise unless you pass `overwrite=True`.

In [ ]:
# === Parameters ===
experiment_name = "houston_enhanced_v1"
description = (
    "Houston enhanced pretraining, baseline hyperparameters from "
    "configs/pretrain/houston_pretrain_enhanced.yaml."
)

# Path to the YAML config to start from.
config_path = "configs/pretrain/houston_pretrain_enhanced.yaml"

# Optional dict-shaped overrides applied on top of the loaded YAML.
# Useful for sweeps without touching the config file.
# Example: {"pretrain": {"lr": 3e-4, "band_mask_ratio": 0.8}}
overrides = {}

# Resume from a checkpoint inside this experiment (or another).
# Pass an absolute or repo-relative path, or None to start fresh.
resume = None

overwrite = False

In [ ]:
import os, sys
from pathlib import Path

# Make repo root importable regardless of where Jupyter was launched.
REPO = Path.cwd()
while not (REPO / "lib" / "experiments.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("Could not locate repo root containing lib/experiments.py")
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Repo root:", REPO)

In [ ]:
from lib.pretrain_runner import run_pretrain

exp = run_pretrain(
    name=experiment_name,
    description=description,
    config=config_path,
    overrides=overrides,
    resume=resume,
    overwrite=overwrite,
)
print("Experiment dir:", exp.root)

In [ ]:
# Peek at the recorded metadata + final history.
import json
print(json.dumps(exp.metadata, indent=2, default=str))

In [ ]:
# List of checkpoints written by this run.
for p in sorted(exp.checkpoints_dir.glob("*.pth")):
    print(p.name, f"{p.stat().st_size/1e6:.1f} MB")